# 05 — mmWave Studio Connection Control

This notebook demonstrates the Python API workflow for establishing a
connection to the AWR2944EVM through mmWave Studio.

**Architecture:** Python/Jupyter is the control layer. mmWave Studio is
the execution backend. Python generates Lua scripts that the user runs
in the mmWave Studio Lua Shell.

In [ ]:
from pathlib import Path
from awr2944_dca.api.experiment import Experiment

## 1. Open or Create an Experiment

In [ ]:
# Open an existing experiment (or create one with Experiment.init)
# exp = Experiment.init('my_connection_test', preset='first-capture')
exp = Experiment.open('.')  # assumes you're in an experiment directory
print(f'Experiment root: {exp.root_dir}')

## 2. Scan COM Ports

In [ ]:
from awr2944_dca.hardware.ports import scan_ports, resolve_port

ports = scan_ports()
for p in ports:
    print(f'{p.com:8s} | {p.friendly_name:40s} | role={p.likely_role} ({p.confidence})')

## 3. Resolve and Save the RS232 Port

In [ ]:
candidates = resolve_port('awr-rs232')
print('Top candidates:')
for c in candidates[:3]:
    print(f'  {c.com} — {c.friendly_name} ({c.confidence})')

In [ ]:
from awr2944_dca.hardware.ports import save_local_hardware

# Save your chosen COM port (change COM6 to your actual port)
chosen_com = 'COM6'
cfg_path = save_local_hardware(exp.root_dir, 'awr-rs232', chosen_com)
print(f'Saved to {cfg_path.name}')

## 4. Generate the Connection Script

In [ ]:
from awr2944_dca.mmws.bridge import ManualOneShotBridge
from awr2944_dca.mmws.models import ConnectionTabConfig

config = ConnectionTabConfig(rs232_com=chosen_com)
log_dir = exp.root_dir / 'ti' / 'probe_logs'
bridge = ManualOneShotBridge(log_dir)

script = bridge.generate_connection_script(config.com_number)
print(f'Generated: {script.name}')
print(f'\nRun this in mmWave Studio Lua Shell:')
print(f'  dofile([[{script.resolve()}]])')

## 5. Check Connection Status

After running the script in mmWave Studio, check the result:

In [ ]:
from awr2944_dca.mmws.stages import StageName

status, result = bridge.check_status(StageName.CONNECTION_ONLY)
print(f'Status: {status.value}')
if result:
    print(f'  COM: {result.get("com_display")}')
    print(f'  Connect return: {result.get("connect_return")}')
    print(f'  Error: {result.get("error")}')